# Time-decompression relay
Fires pulses of fluid through a PT column comprising pyDEW reactants.

In [1]:
import numpy as np
import pyDEW
import os
from thermoengine import model
from copy import copy
from copy import deepcopy
import matplotlib.pyplot as plt
import pandas as pd
import linecache
import cmcrameri.cm as cmc
from scipy.interpolate import LinearNDInterpolator
from timeit import default_timer as timer
import warnings

# pyDEW flags up a few things when the system is created - this counteracts it
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None

# These are the additional functions which we need to work with pyDEW inputs/outputs
import pyDEW_functions as pDF

if not os.path.exists("./data"):
    os.makedirs("./data")


## A1. Preliminaries: mantle column
First we are going to establish the conditions of our column.

In [2]:
# This sets how many steps are taken from the base of the stack to the top
steps = 20

# Create list of pressures
pressures = np.linspace(10, 15, steps+1)  # in kbar
pressures = pressures[::-1]  # invert for the temperatures (so we start at the base of the stack)
pressures = [i*1000 for i in pressures]  # in bar

# Create list of temperatures
temperatures = np.linspace(500, 600, steps+1)  # in degC
temperatures = [i+273.15 for i in temperatures]  # in K


Next we are going to set up our pyDEW system.

In [3]:
# Adjust here the basis species in the DEW system - deepcopy to avoid mucking up defaults
defaultBasisSpecies = pyDEW.defaultsystem.basis_species_names
basisSpecies = deepcopy(defaultBasisSpecies)

defaultOtherSpecies = pyDEW.defaultsystem.other_species_names
otherSpecies = deepcopy(defaultOtherSpecies)

# Add missing Mg complex to the other species
otherSpecies.append('MG(OH)2(AQ)')

# Switch basis species - replace CO2(AQ) with CO3-2 directly to avoid pyDEW issues
basisSpecies[basisSpecies.index('CO2(AQ)')] = 'CO3-2'

# Create DEW system
dew_system = pyDEW.System(basis_species=basisSpecies, other_species=otherSpecies)

# pyDEW minutia - to match Guillaume's setups
# Suppress H2CO3(AQ)
nxmods = [['H2CO3(AQ)', 0, -1, 0]]


We will set our column to have the same composition throughout. This is going to be a list of list of reactants (which is a list of dictionaries). For each reaction, we are going to add a new list of reactants to the overall list.

In [4]:
# List of reactants formed from mineral dictionaries
olvDict = {'reactant': 'OLIVINE(SS)',
           'moles': 5.9,
           'composition': {'FORSTERITE': 0.9, 'FAYALITE': 0.1}}

opxDict = {'reactant': 'ORTHOPYROXENE(SS)',
           'moles': 1.9,
           'composition': {'ENSTATITE_OR': 0.9, 'FERROSILITE': 0.1}}

cpxDict = {'reactant': 'CLINOPYROXENE(SS)',
           'moles': 0.2,
           'composition': {'DIOPSIDE': 0.9, 'HEDENBERGITE': 0.1}}

spnDict = {'reactant': 'SPINEL',
           'moles': 0.04}

startingReactants = [olvDict, opxDict, cpxDict, spnDict]

# Quickly check that our starting fluid-rock ratio (F/R) is going to be 1 (or whatever it is)
startingReactants = pDF.reactantToFR(startingReactants, 1.0, dew_system)

# Create our preliminary compositional stack
startingComposition = pDF.reactantToDF(startingReactants, dew_system)
startingComposition = pd.concat([startingComposition]*len(pressures))
startingComposition['Pressure (bars)'] = pressures
startingComposition['Temperature (K)'] = temperatures
startingComposition['Time'] = [0]*len(pressures)

## A2. Fluid composition
Import the converged EQ3 solutions of G. Siron. This will be the basis for interpolation to get slab fluid compositions.

Create our input fluid and fO2:

In [5]:
# New function here to find a fluid from the database which falls in our 3D space.
def createHybridFluid(proportions, T, P, database='EQ3_pickups_GS.xlsx',
                      sheetNames=['DMM','MORB','Sediments']):
    """
    Function to create a hybrid linearly interpolated fluid from fluid endmember compositions.
    Fluid assumed to be linearly proportional of endmembers.

    Inputs
    ------
    proportions : list of floats
        proportions each fluid endmember contributes to the final fluid
    T : float
        temperature for interpolation, in K
    P : float
        pressure for interpolation, in bars
    database : string
        xlsx database with fluid endmember compositions. Default is 'EQ3_pickups_GS.xlsx'
    sheetNames : list of strings
        names of sheets in xlsx database corresponding to fluid endmembers. Default is
        ['DMM','MORB','Sediments'], corresponding to 'EQ3_pickups_GS.xlsx'

    Outputs
    -------
    exportDict : dictionary
        dictionary describing elemental composition of hybridised fluid
    exportfO2 : float
        float describing hybridised oxygen fugacity
    """

    assert len(proportions) == len(sheetNames), 'check same item proportions and database'

    for i in range(len(proportions)):

        # To store our new fluid
        if i == 0:
            molalitiesDict = {}
        key = sheetNames[i]
        LithData = pd.read_excel(database, sheet_name=sheetNames[i])
        proportion = proportions[i]/np.sum(proportions)

        # Find the species comprising the fluid
        columnHeaders = list(LithData.columns)
        speciesHeaders = columnHeaders[2:]

        # Extract pressures and temperatures
        dataPressures = LithData['P (kbar)'].values
        dataTemperatures = LithData['T (degC)'].values

        # This iterates over all species and linearly interpolates across them based on the
        # molalities pickup sheet
        for j in range(len(speciesHeaders)):
            speciesData = LithData[speciesHeaders[j]]

            # Makes sure that the species aren't all constant, which makes the interpolation wacky
            if speciesData.tolist() == [speciesData.tolist()[0]]*len(speciesData):
                if i == 0:
                    molalitiesDict[speciesHeaders[j]] = speciesData[0] * proportion
                else:
                    molalitiesDict[speciesHeaders[j]] = molalitiesDict[speciesHeaders[j]] + (speciesData[0] * proportion)

            # Otherwise we set up the interpolator for the species given pressure and temperature
            else:
                interp = LinearNDInterpolator(list(zip(dataTemperatures.tolist(), dataPressures.tolist())), speciesData.tolist())

                # Reconvert back to original units here
                interpMolal = interp(T-273.15, P/1000)
                if i == 0:
                    molalitiesDict[speciesHeaders[j]] = float(interpMolal) * proportion
                elif speciesHeaders[j] not in molalitiesDict.keys():
                    molalitiesDict[speciesHeaders[j]] = float(interpMolal) * proportion
                else:
                    molalitiesDict[speciesHeaders[j]] = molalitiesDict[speciesHeaders[j]] + (float(interpMolal) * proportion)

    exportDict = {'NA+': molalitiesDict['NA'],
                  'K+' : molalitiesDict['K'],
                  'CA+2': molalitiesDict['CA'],
                  'MG+2': molalitiesDict['MG'],
                  'AL+3': molalitiesDict['AL'],
                  'H4SIO4(AQ)': molalitiesDict['SI'],
                  'CO3-2' : molalitiesDict['C'],
                  'CL-': molalitiesDict['CL'],
                  'FE+2': molalitiesDict['FE']}
    exportfO2 = molalitiesDict['fO2']
    return exportDict, exportfO2


In [17]:
# Get starting PT for interpolation
sourcePressure = pressures[0]
sourceTemperature = temperatures[0]


## B. Time-decompression relay function

In [20]:
def decompressionRelay(startingColumn, startingMolalities, startingfO2, system, startingFRRatio=1.0, timesteps=10,
                       exportPath='data/bigtest', outputToRead='working/output', tabToRead='working/tab',
                       FRRatioOption0=None, FRRatioOption1=None):
    """
    Decompression relay as a function of time. This models a column of rock getting flushed with a fluid periodically.
    Definitions:
    - initial : before the reaction at each stage
    - final : after the reaction at each stage
    - 1 kg : relative to 1 kg of fluid
    - running : relative to the overall fluid

    INPUTS
    ------
    startingColumn : pandas.DataFrame
        pandas DataFrame with each column a different mineral concentration (moles), with two additional columns
        'Pressure (bars)' and 'Temperature (K)'
    startingMolalities : dictionary
        dictionary of elemental molalities in log molal describing the composition of the flushing fluid
    system : pyDEW.System object
        pyDEW.System for calculations
    timesteps : integer
        number of times to flush our solid with the fluid
    exportPath : string
        path to export xlsx to
    outputToRead : string
        path to pyDEW.Reaction output file
    tabToRead : string
        path to pyDEW.Reaction tab file
    FRRatioOption0 : None, 'fluid', 'fixed', 'limit'
        first batch of options for fluid-rock ratio
    FRRatioOption1 : float
        fluid-rock ratio (dependent on FRRatioOption0)

    OUTPUTS
    -------
    xlsx file
        Excel file with solid, element, and fluid speciation data

    """

    start = timer()
    t = 0

    # The headers for our output file may contain solid-solution minerals and their endmembers
    # We need to set the headers up now so we can allow for the mineral endmembers and some other options
    acceptableHeaders = [i.props['phase_name'] for i in system.minerals]
    acceptableHeaders.extend(['Pressure (bars)', 'Temperature (K)', 'Time', 'max log zi',
                              'Mass (g)', 'Volume (cc)', 'Density (g/cc)', 'Crash flag'])
    acceptableHeaders.extend(['F/R before reaction', 'F/R after reaction', 'Running F/R', 'Initial mass of solid',
                              'Initial mass of fluid', 'Final mass of solid (r. 1 kg)', 'Final mass of fluid (r. 1 kg)',
                              'Final mass of solid', 'Final mass of fluid'])
    acceptableHeaders.extend(['DIOPSIDE(CPX)', 'JADEITE(CPX)', 'HEDENBERGITE(CPX)'])
    acceptableHeaders.extend(['KFELDSPAR(KFSP)', 'ALBITE(KFSP)'])

    ###############################################################################################################################
    ##### TIME LOOP #####
    print('Time loop started')
    while t < timesteps:
        if t == 0:
            workingColumn = startingColumn
        else:
            workingColumn = lastColumn

        # Storage for running F/R throughout column, and for tracking solid masses and volumes
        initialFRRatioList = []  # This is keeping track of the F/R before each stage of reaction
        finalFRRatio1kgList = []  # This is keeping track of the F/R after each reaction relative to 1 kg fluid
        runningFRRatioList = []  # This is keeping track of the F/R after each stage of reaction

        # Mass of the solid/fluid BEFORE the reaction
        initialSolidMassList = []
        initialFluidMassList = []

        # Mass of the solid before the reaction, relative to 1 kg initial fluid
        initialSolidMass1kgList = []

        # Mass of the solid/fluid after the reaction, relative to 1 kg initial fluid
        finalFluidMass1kgList = []
        finalSolidMass1kgList = []

        # Mass of the solid/fluid AFTER the reaction
        finalFluidMassList = []
        finalSolidMassList = []

        # This is for keeping track of crashes that will trigger the UserError try clause below
        # If True, then the previous reaction has failed
        crashFlag = False

        ###########################################################################################################################
        ##### DECOMPRESSION RELAY #####
        for n in range(len(startingColumn)):

            #######################################################################################################################
            #### ESTABLISH REACTANT CONDITIONS ####
            workingParameters = workingColumn.iloc[n]
            workingReactants = pDF.DFtoReactant(workingParameters, system)
            workingT = workingParameters['Temperature (K)']
            workingP = workingParameters['Pressure (bars)']

            #######################################################################################################################
            #### MODIFY FLUID-ROCK RATIO OPTIONS ####
            # Fluid-rock ratio (F/R or FR) is the mass of fluid divided by the mass of reactants
            # The initial mass of rock is multiplied by a value so that the amount of fluid at 1 kg interacts with an
            # appropriate amount of rock given a starting F/R. By using the same value on the products, we can determine
            # the mass of solid remaining after the reaction. 
            # e.g., if 1 kg of rock interacts with 0.5 kg of fluid, then the initial calculation is 2 kg of rock with
            # 1 kg of fluid. After, we may end up with, e.g., 2.1 kg of resultant rock and 0.9 kg fluid. We will correct this
            # by applying the same coefficient to end up with 1.05 kg of resultant rock and 0.45 kg fluid.


            # startingFRRatio : F/R at n=0 (or at t=0, n=0 when FRRationOption0 == 'fixedStart')
            # initialFRRatio : F/R before reaction
            # runningFRRatio : 'true' F/R (relative to starting F/R)
            # solidModulate : the amount to multiply our product by to return back to the correct amount of solid, therefore the amount to
            #                 divide our initial solid by to get the reactant relative to 1 kg fluid

            initialReactantMass = pDF.findReactantMass(workingReactants, system)

            # If the previous reaction crashed, we just continue using the same initial ratios/masses as the previous stage
            if crashFlag:
                continue

            # 'None' : we want to keep the F/R the same at every step - this will essentially increase the overall F/R from bottom to top
            # of the column, which is unrealistic
            elif FRRatioOption0 == None:
                initialFRRatio = startingFRRatio
                initialFluidMass = initialReactantMass * initialFRRatio
            else:
                if n == 0:
                    # 'fluid' : we want to change the F/R during the chain of reactions, but keep the F/R at n=0 the same throughout
                    if FRRatioOption0 == 'fluid':
                        initialFRRatio = startingFRRatio
                        initialFluidMass = initialReactantMass * initialFRRatio
                    # 'fixedStart' : we want to change the F/R during the chain of reactions but keep the mass of fluid at n=0 the same
                    # This fluid mass is determined from the starting F/R
                    elif FRRatioOption0 == 'fixedStart':
                        if t == 0:
                            initialFluidMass = initialReactantMass * startingFRRatio
                            fixedInitialFluidMass = initialReactantMass * startingFRRatio
                            initialFRRatio = startingFRRatio
                        else:
                            initialFluidMass = fixedInitialFluidMass
                            initialFRRatio = fixedInitialFluidMass / initialReactantMass
                # If n > 0, then initial fluid mass decreases by a ratio determined by the mass change recorded via each reaction
                else:
                    initialFluidMass = initialFluidMass * pDF.findFluidMassChange(tabToRead)
                    initialFRRatio = initialFluidMass / initialReactantMass
                workingReactants1kg = pDF.reactantToFR(workingReactants, initialFRRatio, system)
                initialReactant1kgMass = pDF.findReactantMass(workingReactants1kg, system)
                solidModulate = initialFluidMass / 1000.0

            # These lists are just to store information on the initial reaction conditions
            initialFRRatioList.append(initialFRRatio)
            initialSolidMassList.append(initialReactantMass)
            initialFluidMassList.append(initialFluidMass)
            initialSolidMass1kgList.append(initialReactant1kgMass)


            #######################################################################################################################
            #### ESTABLISH FLUID CONDITIONS ####
            # If first, then use the slab fluid. Else use the fluid from the previous calculation
            if n == 0:
                workingFluid = pyDEW.Fluid(system, workingT, workingP, molalities=startingMolalities, fO2=startingfO2,
                                           uebal='H+', uacion='CL-', nxmods=nxmods, aH2O_mode='molfraction')

                # Also adjust our working reactants by the appropriate F/R
                workingReactants = pDF.reactantToFR(workingReactants, startingFRRatio, system)

            else:
                # If no mineral product was found previously, use the previous lastElements/workingfO2 (i.e., crashFlag==True)
                # Else, update with the values from the calculation prior to the last one (i.e., crashFlag==False)
                if crashFlag == False:
                    lastElements = dewReaction.elements.iloc[-1]
                    workingfO2 = dewReaction.fO2[-1]

                molalities = {'NA+': lastElements['Na'],
                              'CA+2': lastElements['Ca'],
                              'MG+2': lastElements['Mg'],
                              'AL+3': lastElements['Al'],
                              'H4SIO4(AQ)': lastElements['Si'],
                              'CO3-2' : lastElements['C'],
                              'CL-': lastElements['Cl'],
                              'FE+2': lastElements['Fe']}

                # pyDEW returns log molalities, whereas the inputs are in molalities
                molalities.update((i, 10**j) for i, j in molalities.items())
                workingFluid = pyDEW.Fluid(system, workingT, workingP, molalities=molalities, fO2=workingfO2,
                                           uebal='H+', uacion='CL-', nxmods=nxmods, aH2O_mode='molfraction')

            #######################################################################################################################
            #### RUN REACTION ####
            # Calculate the reaction here between reactants and products. The output will be a series of pandas dataframes
            try:
                # If the reaction goes ahead without issue, then perform the reaction with the given conditions
                dewReaction = pyDEW.Reaction(workingFluid, workingT, workingP, workingReactants, zimax=5.0, calculation_mode='titration')

                #### CREATE NEW DATAFRAMES FOR OUTPUT ####
                # Get the solids formed during this reaction (from 1 kg fluid)
                # This doesn't work if there is no solid product, which is why this is all nestled in a big try clause
                lastReactionDF = pDF.findSolidComposition(outputToRead, system, workingP, workingT)
                maxLogZi = lastReactionDF['log zi'].iloc[-1]  # Get maximum reaction progress

                # Get the dataframe representing the last solid (from 1 kg fluid)
                lastSolid1kg = lastReactionDF[lastReactionDF['log zi'] == maxLogZi]
                lastSolid1kgHeaders = lastSolid1kg['phase/end-member'].tolist()
                lastSolid1kgValues = lastSolid1kg['moles'].tolist()

                # Because CPX has two different solid solutions, need to clarify the differences or pandas throws a fit
                if 'CLINOPYROXENE(SS)' in lastSolid1kgHeaders:
                    index = lastSolid1kgHeaders.index('CLINOPYROXENE(SS)')
                    lastSolid1kgHeaders[index+1] = 'DIOPSIDE(CPX)'
                    lastSolid1kgHeaders[index+2] = 'JADEITE(CPX)'
                    lastSolid1kgHeaders[index+3] = 'HEDENBERGITE(CPX)'

                # Same thing for k-spar
                if 'KFELDSPAR(SS)' in lastSolid1kgHeaders:
                    index = lastSolid1kgHeaders.index('KFELDSPAR(SS)')
                    lastSolid1kgHeaders[index+1] = 'KFELDSPAR(KFSP)'
                    lastSolid1kgHeaders[index+2] = 'ALBITE(KFSP)'

                ###################################################################################################################
                #### FORMAT SOLID RELATIVE TO 1 KG FLUID ####
                # Create new dataframe and reactant object for our final reactant (from 1 kg fluid)
                lastSolid1kgDF = pd.DataFrame([lastSolid1kgValues])
                lastSolid1kgDF.columns = lastSolid1kgHeaders
                lastSolid1kgDF = lastSolid1kgDF[lastSolid1kgDF.columns.intersection(acceptableHeaders)]  # Remove mineral names with solid solution

                # There is an issue where dataframes are concated together all columns duplicate except one which is near-zero. This should fix this
                if len(lastSolid1kgDF.columns) != len(set(lastSolid1kgDF.columns)):
                    lastSolid1kgDF = lastSolid1kgDF.iloc[:, ~lastSolid1kgDF.columns.duplicated()]

                # Convert our pandas output into a reactant list of dictionaries
                lastSolid1kgReactant = pDF.DFtoReactant(lastSolid1kgDF.iloc[0], system)

                # Determine the physical properties of the reactant at the current PT relative to 1 kg fluid
                lastSolid1kgMass = pDF.findReactantMass(lastSolid1kgReactant, system)
                lastSolid1kgVolume = pDF.findReactantVolume(lastSolid1kgReactant, system, workingP, workingT)
                lastSolid1kgDensity = lastSolid1kgMass / lastSolid1kgVolume

                # So far everything has been done relative to 1 kg of fluid. Here we need to recalculate the fluid and solid masses.
                # Get the final mass of fluid and solid
                lastFluid1kgMass = 1000.0 * pDF.findFluidMassChange(tabToRead)  # Mass of the fluid at the end of the reaction (relative to initial 1 kg)
                finalFRRatio = lastFluid1kgMass / lastSolid1kgMass
                if n == 0:
                    runningFRRatio = initialFRRatio * finalFRRatio
                else:
                    runningFRRatio = runningFRRatio * finalFRRatio
                finalFluidMass1kgList.append(lastFluid1kgMass)  # This is the mass of the fluid relative to 1 kg initial
                finalSolidMass1kgList.append(lastSolid1kgMass)  # This is the 'true' mass of the solid relative to 1 kg initial
                finalFRRatio1kgList.append(finalFRRatio)  # This is the F/R after this stage of reaction
                runningFRRatioList.append(runningFRRatio)  # THis is the running F/R relative to the initial fluid at n = 0

                ###################################################################################################################
                #### FORMAT SOLID RELATIVE TO INITIAL FLUID MASS (ADJUSTED) ####
                # Multiply the product solid values by the modulate value to get the true solid mass (i.e., for whatever amount of fluid there is
                # before correction to 1 kg for F/R). Then do the same as above for each corrected solid
                adjustedLastSolidReactant = []
                for j in lastSolid1kgReactant:
                    j['moles'] = solidModulate * j['moles']
                    adjustedLastSolidReactant.append(j)
                adjustedLastSolidDF = pDF.reactantToDF(adjustedLastSolidReactant, system)
                adjustedLastSolidDF.reset_index().T.drop_duplicates().T
                adjustedLastFluidMass = lastFluid1kgMass * solidModulate

                # Determine the 'true' physical properties of the solid
                adjustedLastSolidMass = pDF.findReactantMass(adjustedLastSolidReactant, system)
                adjustedLastSolidVolume = pDF.findReactantVolume(adjustedLastSolidReactant, system, workingP, workingT)
                adjustedLastSolidDensity = adjustedLastSolidMass / adjustedLastSolidVolume
                finalFluidMassList.append(adjustedLastFluidMass)
                finalSolidMassList.append(adjustedLastSolidMass)

                ###################################################################################################################
                #### FORMAT FLUID ####
                # Set up the information for the fluid of the last reaction
                lastReactionElementsHeaders = dewReaction.elements.columns
                lastReactionElements = dewReaction.elements.iloc[-1].tolist()
                lastReactionSpeciesHeaders = dewReaction.species_concs.columns
                lastReactionSpecies = dewReaction.species_concs.iloc[-1].tolist()

                # Save the data for the next iteration in case it fails
                backupSolid1kgDF = deepcopy(lastSolid1kgDF)
                backupAdjustedSolidDF = deepcopy(adjustedLastSolidDF)
                backupElements = deepcopy(lastReactionElements)
                backupFluid = deepcopy(lastReactionSpecies)

                # Undo the crashFlag if it has been triggered
                crashFlag = False

            #######################################################################################################################
            except KeyError:
                # If the resultant dataframe is empty, then no reaction has happened and we should use the previous input as the output
                print('Calculation at [n=' + str(n) +', t=' + str(t) + '] has failed: using results from previous n')
                maxLogZi = -999.0

                # This is in case there is no reaction at the start - this has not been encountered yet so I don't know if this works
                if n == 0:
                    print('Calculation at n=0 has failed; retaining solid composition and moving fluid along')
                    lastSolid1kgDF = pDF.reactantToDF(workingReactants, system)
                    adjustedLastSolidReactant = []
                    for j in workingReactants:
                        j['moles'] = solidModulate * j['moles']
                        adjustedLastSolidReactant.append(j)
                    adjustedLastSolidDF = pDF.reactantToDF(adjustedLastSolidReactant, system)

                    # Get the elements and reformat
                    lastReactionElementsHeaders = workingFluid.elemental_comp['element'].tolist()
                    lastReactionElementMolal = workingFluid.elemental_comp['molality'].tolist()
                    lastReactionElementLogMolal = [[np.log10(i)] for i in lastReactionElementMolal]
                    lastReactionElements = pd.DataFrame(lastReactionElementLogMolal)
                    lastReactionElements.columns = lastReactionElementsHeaders

                    # Get the species and reformat
                    lastReactionSpeciesHeaders = workingFluid.aqueous_species['species'].tolist()
                    lastReactionSpeciesMolal = workingFluid.aqueous_species['molality'].tolist()
                    lastReactionSpeciesLogMolal = [[np.log10(i)] for i in lastReactionSpeciesMolal]
                    lastReactionSpecies = pd.DataFrame(lastReactionSpeciesLogMolal)
                    lastReactionSpecies.columns = lastReactionSpeciesHeaders

#                     # Get the physical properties of the solid
#                     lastSolidMass = findReactantMass(workingReactants, system)
#                     lastSolidVolume = findReactantVolume(workingReactants, system, workingP, workingT)
#                     lastSolidDensity = lastSolidMass / lastSolidVolume

#                     adjustedLastSolidMass = pDF.findReactantMass(adjustedLastSolidReactant, system)
#                     adjustedLastSolidVolume = pDF.findReactantVolume(adjustedLastSolidReactant, system, workingP, workingT)
#                     adjustedLastSolidDensity = adjustedLastSolidMass / adjustedLastSolidVolume

                # Otherwise, we can use our backups - the physical properties are already calculated
                else:
                    lastSolid1kgDF = backupSolid1kgDF
                    adjustedLastSolid1kgDF = backupAdjustedSolidDF
                    adjustedLastSolidReactant = pDF.DFtoReactant(adjustedLastSolidDF.iloc[0], system)
                    lastReactionElements = backupElements
                    lastReactionSpecies = backupFluid

                # Convert reactants
                lastSolid1kgReactant = pDF.DFtoReactant(lastSolid1kgDF.iloc[0], system)

                # Now append things to the lists
                # Relative to 1 kg initial fluid
                lastSolid1kgMass = pDF.findReactantMass(lastSolid1kgReactant, system)
                lastSolid1kgVolume = pDF.findReactantVolume(lastSolid1kgReactant, system, workingP, workingT)
                lastSolid1kgDensity = lastSolid1kgMass / lastSolid1kgVolume
                lastFluid1kgMass = finalFluidMass1kgList[-1]  # But this does not account for expansion effects due to PT changes
                finalFRRatio = lastFluid1kgMass / lastSolid1kgMass
                runningFRRatio = runningFRRatio * finalFRRatio
                finalFluidMass1kgList.append(lastFluid1kgMass)  # This is the mass of the fluid relative to 1 kg initial
                finalSolidMass1kgList.append(lastSolid1kgMass)  # This is the 'true' mass of the solid relative to 1 kg initial
                finalFRRatio1kgList.append(finalFRRatio)  # This is the F/R after this stage of reaction
                runningFRRatioList.append(runningFRRatio)  # THis is the running F/R relative to the initial fluid at n = 0

                # Relative to initial fluid
                adjustedLastSolidMass = pDF.findReactantMass(adjustedLastSolidReactant, system)
                adjustedLastSolidVolume = pDF.findReactantVolume(adjustedLastSolidReactant, system, workingP, workingT)
                adjustedLastSolidDensity = adjustedLastSolidMass / adjustedLastSolidVolume
                adjustedLastFluidMass = lastFluid1kgMass * solidModulate
                finalFluidMassList.append(adjustedLastFluidMass)
                finalSolidMassList.append(adjustedLastSolidMass)

                # Flip the flag up so the code recognises that a crash has happened and can react accordingly above
                crashFlag = True

            # Create DataFrames for solid, fluid elements, fluid species
            if n == 0:
                # This should be the solid at the given stage/time
                timeSolid1kgDF = deepcopy(lastSolid1kgDF)
                timeSolid1kgDF['Pressure (bars)'] = [workingP]
                timeSolid1kgDF['Temperature (K)'] = [workingT]
                timeSolid1kgDF['Time'] = [t + 1]
                timeSolid1kgDF['max log zi'] = [maxLogZi]
                timeSolid1kgDF['Mass (g)'] = [lastSolid1kgMass]
                timeSolid1kgDF['Volume (cc)'] = [lastSolid1kgVolume]
                timeSolid1kgDF['Density (g/cc)'] = [lastSolid1kgDensity]
                timeSolid1kgDF['Crash flag'] = [crashFlag]

                # This should be adjusted for total mass of the rock
                allSolidDF = deepcopy(adjustedLastSolidDF)
                allSolidDF['Pressure (bars)'] = [workingP]
                allSolidDF['Temperature (K)'] = [workingT]
                allSolidDF['Time'] = [t + 1]
                allSolidDF['max log zi'] = [maxLogZi]
                allSolidDF['Mass (g)'] = [adjustedLastSolidMass]
                allSolidDF['Volume (cc)'] = [adjustedLastSolidVolume]
                allSolidDF['Density (g/cc)'] = [adjustedLastSolidDensity]
                allSolidDF['Crash flag'] = [crashFlag]

                timeElementsDF = pd.DataFrame([lastReactionElements])
                timeElementsDF.columns = lastReactionElementsHeaders
                timeElementsDF['Pressure (bars)'] = [workingP]
                timeElementsDF['Temperature (K)'] = [workingT]
                timeElementsDF['Time'] = [t + 1]

                timeFluidDF = pd.DataFrame([lastReactionSpecies])
                timeFluidDF.columns = lastReactionSpeciesHeaders
                timeFluidDF['Pressure (bars)'] = [workingP]
                timeFluidDF['Temperature (K)'] = [workingT]
                timeFluidDF['Time'] = [t + 1]

            # Else append the new data to the previous step
            else:
                tempSolid1kgDF = deepcopy(lastSolid1kgDF)
                tempSolid1kgDF['Pressure (bars)'] = [workingP]
                tempSolid1kgDF['Temperature (K)'] = [workingT]
                tempSolid1kgDF['Time'] = [t + 1]
                tempSolid1kgDF['max log zi'] = [maxLogZi]
                tempSolid1kgDF['Mass (g)'] = [lastSolid1kgMass]
                tempSolid1kgDF['Volume (cc)'] = [lastSolid1kgVolume]
                tempSolid1kgDF['Density (g/cc)'] = [lastSolid1kgDensity]
                tempSolid1kgDF['Crash flag'] = [crashFlag]
                timeSolid1kgDF = pd.concat([timeSolid1kgDF, tempSolid1kgDF])

                tempAllSolidDF = deepcopy(adjustedLastSolidDF)
                tempAllSolidDF['Pressure (bars)'] = [workingP]
                tempAllSolidDF['Temperature (K)'] = [workingT]
                tempAllSolidDF['Time'] = [t + 1]
                tempAllSolidDF['max log zi'] = [maxLogZi]
                tempAllSolidDF['Mass (g)'] = [adjustedLastSolidMass]
                tempAllSolidDF['Volume (cc)'] = [adjustedLastSolidVolume]
                tempAllSolidDF['Density (g/cc)'] = [adjustedLastSolidDensity]
                allSolidDF = pd.concat([allSolidDF, tempAllSolidDF])

                tempElementsDF = pd.DataFrame([lastReactionElements])
                tempElementsDF.columns = lastReactionElementsHeaders
                tempElementsDF['Pressure (bars)'] = [workingP]
                tempElementsDF['Temperature (K)'] = [workingT]
                tempElementsDF['Time'] = [t + 1]
                timeElementsDF = pd.concat([timeElementsDF, tempElementsDF])

                tempFluidDF = pd.DataFrame([lastReactionSpecies])
                tempFluidDF.columns = lastReactionSpeciesHeaders
                tempFluidDF['Pressure (bars)'] = [workingP]
                tempFluidDF['Temperature (K)'] = [workingT]
                tempFluidDF['Time'] = [t + 1]
                timeFluidDF = pd.concat([timeFluidDF, tempFluidDF])

        #### REMOVE BAD HEADERS ####
        # The solid-solution series may be recorded, in which case we remove the mineral in question and use the endmembers
        timeSolid1kgDF = timeSolid1kgDF[timeSolid1kgDF.columns.intersection(acceptableHeaders)]
        timeSolid1kgDF['F/R before reaction'] = initialFRRatioList
        timeSolid1kgDF['F/R after reaction'] = finalFRRatio1kgList
        timeSolid1kgDF['Running F/R'] = runningFRRatioList
        timeSolid1kgDF['Final mass of solid (r. 1 kg)'] = finalSolidMass1kgList
        timeSolid1kgDF['Final mass of fluid (r. 1 kg)'] = finalFluidMass1kgList
        timeSolid1kgDF['Final mass of solid'] = finalSolidMassList
        timeSolid1kgDF['Final mass of fluid'] = finalFluidMassList

        #### COMBINE TIMESTEP DATAFRAME WITH TOTAL DATAFRAME ####
        if t == 0:
            totalSolid1kgDF = timeSolid1kgDF
            totalAllSolidDF = allSolidDF
            totalElementsDF = timeElementsDF
            totalFluidDF = timeFluidDF
        else:
            totalSolid1kgDF = pd.concat([totalSolid1kgDF, timeSolid1kgDF])
            totalAllSolidDF = pd.concat([totalAllSolidDF, allSolidDF])
            totalElementsDF = pd.concat([totalElementsDF, timeElementsDF])
            totalFluidDF = pd.concat([totalFluidDF, timeFluidDF])
        lastColumn = deepcopy(allSolidDF)  # Propagate forward the 'true' results
        print('Timestep t=' + str(t) + ' is complete, with ' + str(n*(t+1)) + ' total calculations performed')
        t+=1

    #### EXPORT ####
    # Export to xlsx
    with pd.ExcelWriter(exportPath+'.xlsx') as writer:
        totalSolid1kgDF.to_excel(writer, sheet_name='solid_r1kg')
        totalAllSolidDF.to_excel(writer, sheet_name='solid_true')
        totalElementsDF.to_excel(writer, sheet_name='elements')
        totalFluidDF.to_excel(writer, sheet_name='fluid')
    print('data exported to: ' + exportPath+'.xlsx')
    end = timer()
    print('Total calculations: ' + str(timesteps*len(startingColumn)))
    print('Total runtime: ' + str(round(end-start, 1)) + ' seconds')
    print('Total runtime: ' + str(round((end-start)/60, 1)) + ' minutes')
    return


In [21]:
startingFluid = createHybridFluid([1, 0, 0], sourceTemperature, sourcePressure)
decompressionRelay(startingComposition, startingFluid[0], startingFluid[1],
                   dew_system, timesteps=10,
                   exportPath='data/FRTest1', FRRatioOption0='fluid')

Time loop started
Timestep t=0 is complete, with 20 total calculations performed
Timestep t=1 is complete, with 40 total calculations performed
Timestep t=2 is complete, with 60 total calculations performed
Timestep t=3 is complete, with 80 total calculations performed
Timestep t=4 is complete, with 100 total calculations performed
Timestep t=5 is complete, with 120 total calculations performed
Timestep t=6 is complete, with 140 total calculations performed
Timestep t=7 is complete, with 160 total calculations performed
Timestep t=8 is complete, with 180 total calculations performed
Timestep t=9 is complete, with 200 total calculations performed
data exported to: data/FRTest1.xlsx
Total calculations: 210
Total runtime: 233.4 seconds
Total runtime: 3.9 minutes


In [22]:
startingFluid = createHybridFluid([1, 0, 0], sourceTemperature, sourcePressure)
decompressionRelay(startingComposition, startingFluid[0], startingFluid[1],
                   dew_system, timesteps=10,
                   exportPath='data/FRTest2', FRRatioOption0='fixedStart')

Time loop started
Timestep t=0 is complete, with 20 total calculations performed
Timestep t=1 is complete, with 40 total calculations performed
Timestep t=2 is complete, with 60 total calculations performed
Timestep t=3 is complete, with 80 total calculations performed
Timestep t=4 is complete, with 100 total calculations performed
Timestep t=5 is complete, with 120 total calculations performed
Timestep t=6 is complete, with 140 total calculations performed
Timestep t=7 is complete, with 160 total calculations performed
Timestep t=8 is complete, with 180 total calculations performed
Timestep t=9 is complete, with 200 total calculations performed
data exported to: data/FRTest2.xlsx
Total calculations: 210
Total runtime: 234.0 seconds
Total runtime: 3.9 minutes
